# Load Data and Extract Audio

In [32]:
import subprocess
import os

DataFolder = "Data/TestData1"

mov_path = None
wav_path = None
py_path = None
VIDEO_CLAP_FRAME = None

def extract_audio_from_mov(mov_path, wav_path):
    """
    Helper command to strictly extract .wav standard acoustics natively from Apple .mov containers using OSX afconvert.
    """
    subprocess.run(['afconvert', '-f', 'WAVE', '-d', 'LEI16', mov_path, wav_path])
    print(f"Extraction complete: Saved standalone wave audio to {wav_path}")

for filename in os.listdir(DataFolder):
    if filename.endswith('.mov'):
        mov_path = os.path.join(DataFolder, filename)
        wav_path = os.path.join(DataFolder, filename.replace('.mov', '.wav'))
        py_path = os.path.join(DataFolder, filename.replace('.mov', '.blender.py'))
        frame = os.path.join(DataFolder, "frame.txt")
        VIDEO_CLAP_FRAME = int(open(frame).read().strip())

        extract_audio_from_mov(mov_path, wav_path)
        break

Extraction complete: Saved standalone wave audio to Data/TestData1/ar-captured-1779403063.wav


In [36]:
# %%
import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy.io import wavfile
from scipy.spatial.transform import Rotation, Slerp
import cv2
import plotly.graph_objects as go
import ast
import re
from sklearn.cluster import DBSCAN

# =====================================================================
# 1. PARSING & SYNCHRONIZATION 
# =====================================================================

def parse_blender_tracking_log(py_path, fps=60.0):
    with open(py_path, 'r') as f:
        content = f.read()
    fps_match = re.search(r'bpy\.context\.scene\.render\.fps\s*=\s*(\d+)', content)
    fps = float(fps_match.group(1)) if fps_match else fps
    
    frames = ast.literal_eval(re.search(r'movement_keyframes\s*=\s*(\[.*?\])', content, re.DOTALL).group(1))
    locations = ast.literal_eval(re.search(r'locations\s*=\s*(\[.*?\])', content, re.DOTALL).group(1))
    rotations = ast.literal_eval(re.search(r'rotations\s*=\s*(\[.*?\])', content, re.DOTALL).group(1))
    
    timestamps, translations, rotation_matrices = [], [], []
    for frame, loc, rot in zip(frames, locations, rotations):
        timestamps.append(frame / fps)
        translations.append(loc)
        # Assuming Blender ZXY Euler
        r = Rotation.from_euler('ZXY', [rot[2], rot[0], rot[1]], degrees=False)
        rotation_matrices.append(r.as_matrix())
        
    return pd.DataFrame({'timestamp': timestamps, 'translation': translations, 'rotation_matrix': rotation_matrices})


def parse_blender_intrinsics(py_path):
    with open(py_path, 'r') as f:
        content = f.read()
    width = float(re.search(r'resolution_x\s*=\s*([\d.]+)', content).group(1))
    height = float(re.search(r'resolution_y\s*=\s*([\d.]+)', content).group(1))
    lens = float(re.search(r'lens\s*=\s*([\d.]+)', content).group(1))
    sensor_width = float(re.search(r'sensor_width\s*=\s*([\d.]+)', content).group(1))
    
    fx = (lens / sensor_width) * width
    # Assuming square pixels, fy = fx
    return np.array([[fx, 0, width/2.0], [0, fx, height/2.0], [0, 0, 1]]), int(width), int(height)


def select_adaptive_voxel_threshold(voxel_grid, dense_percentile=95, sparse_percentile=90, density_threshold=2000):
    nonzero_voxels = voxel_grid[voxel_grid > 0]
    nonzero_count = len(nonzero_voxels)

    if nonzero_count == 0:
        print("Adaptive voxel threshold: voxel grid is empty.")
        return None, 0, dense_percentile

    chosen_percentile = sparse_percentile if nonzero_count < density_threshold else dense_percentile
    threshold = np.percentile(nonzero_voxels, chosen_percentile)
    print(
        f"Adaptive voxel threshold: {chosen_percentile}th percentile over {nonzero_count} non-zero voxels -> {threshold:.3f}"
    )
    return threshold, nonzero_count, chosen_percentile

# =====================================================================
# 2. SIGNAL PROCESSING (WITH SNR CONSTRAINT)
# =====================================================================

def compute_acoustic_distances(audio_sig, trajectory_df, sample_rate, reference_chirp, snr_threshold=3.0):
    SPEED_OF_SOUND = 343.0  
    if audio_sig.ndim > 1: audio_sig = audio_sig[:, 0]
    timestamps = trajectory_df['timestamp'].to_numpy()
    distances = np.full(len(timestamps), np.nan)
    window_samples = int(0.100 * sample_rate)
    
    center_indices = (timestamps * sample_rate).astype(int)
    for i, idx_center in enumerate(center_indices):
        if idx_center + window_samples > len(audio_sig): continue
        audio_window = audio_sig[idx_center : idx_center + window_samples]
        if len(audio_window) < len(reference_chirp): continue
            
        xcorr = signal.correlate(audio_window, reference_chirp, mode='valid')
        
        # SNR Calculation Constraint
        noise_floor = np.sqrt(np.mean(audio_window**2)) + 1e-6 
        peaks, _ = signal.find_peaks(xcorr, height=noise_floor * snr_threshold, distance=int(0.001 * sample_rate))
        
        if len(peaks) >= 2:
            delay = (peaks[1] - peaks[0]) / sample_rate
            dist = (SPEED_OF_SOUND * delay) / 2.0
            if dist <= 1.2: 
                distances[i] = dist
    return distances

# =====================================================================
# 3. 3D HOUGH SPACE MAPPING
# =====================================================================

def build_hough_accumulator(pose_df, acoustic_distances, voxel_size=0.04, margin=1.5):
    trajectory = np.array(pose_df['translation'].tolist())
    rotations = np.array(pose_df['rotation_matrix'].tolist())
    min_bounds = np.min(trajectory, axis=0) - margin
    max_bounds = np.max(trajectory, axis=0) + margin
    
    grid_dims = np.ceil((max_bounds - min_bounds) / voxel_size).astype(int)
    voxel_grid = np.zeros(grid_dims, dtype=np.int32)
    
    x_bins = min_bounds[0] + np.arange(grid_dims[0]) * voxel_size
    y_bins = min_bounds[1] + np.arange(grid_dims[1]) * voxel_size
    z_bins = min_bounds[2] + np.arange(grid_dims[2]) * voxel_size
    
    for i, (C_i, d_i) in enumerate(zip(trajectory, acoustic_distances)):
        if np.isnan(d_i) or d_i <= 0: continue
        
        # Local 3D Sub-Grid Bounding Box (Crucial Constraint)
        idx_min = np.maximum(0, np.floor((C_i - (d_i + voxel_size) - min_bounds) / voxel_size)).astype(int)
        idx_max = np.minimum(grid_dims, np.ceil((C_i + (d_i + voxel_size) - min_bounds) / voxel_size)).astype(int)
        
        ix, iy, iz = np.arange(idx_min[0], idx_max[0]), np.arange(idx_min[1], idx_max[1]), np.arange(idx_min[2], idx_max[2])
        if len(ix)==0 or len(iy)==0 or len(iz)==0: continue

        X_s, Y_s, Z_s = np.meshgrid(x_bins[ix], y_bins[iy], z_bins[iz], indexing='ij')
        
        dist_to_cam = np.sqrt((X_s - C_i[0])**2 + (Y_s - C_i[1])**2 + (Z_s - C_i[2])**2)
        forward_world = -rotations[i][:, 2] 
        dot_prod = ((X_s - C_i[0])*forward_world[0] + (Y_s - C_i[1])*forward_world[1] + (Z_s - C_i[2])*forward_world[2]) / (dist_to_cam + 1e-6)
        
        # Increment voxels on the sphere shell within FOV
        voxel_grid[idx_min[0]:idx_max[0], idx_min[1]:idx_max[1], idx_min[2]:idx_max[2]] += (np.abs(dist_to_cam - d_i) < voxel_size) & (dot_prod >= 0.707)
        
    return voxel_grid, min_bounds, voxel_size

# =====================================================================
# 4. RECTANGULAR GEOMETRY EXTRUSION & PROJECTION
# =====================================================================

def generate_balanced_room_mesh(
    voxel_points,
    trajectory_df,
    residual_threshold=0.08,
    resolution=0.03,
    dbscan_eps=None,
    dbscan_min_samples=4,
    cluster_keep_ratio=0.35,
    max_clusters_to_keep=4,
    return_debug=False,
):
    """
    Uses Spatial RANSAC purely as a noise filter to extract the dense wall cores.
    Then applies a global Minimum Area Rectangle to the clean points to find a 
    perfectly balanced, orthogonal 4-wall room without 'first-wall bias'.
    """
    debug_info = {
        'input_point_count': int(len(voxel_points)),
        'ransac_point_count': 0,
        'dbscan_cluster_count': 0,
        'dbscan_noise_count': 0,
        'kept_point_count': 0,
        'rect_width': 0.0,
        'rect_depth': 0.0,
    }

    if len(voxel_points) == 0:
        print("generate_balanced_room_mesh input point count: 0")
        if return_debug:
            return np.empty((0, 3)), debug_info
        return np.empty((0, 3))

    points_2d = np.float32(voxel_points[:, [0, 2]])
    print(f"generate_balanced_room_mesh input point count: {len(points_2d)}")
    
    # --- 1. RANSAC DENOISER ---
    clean_wall_points = []
    pts_pool = points_2d.copy()
    
    for _ in range(10): # Hunt for all possible dense clusters
        if len(pts_pool) < 20: break
            
        best_inliers = []
        for _ in range(500):
            idx = np.random.choice(len(pts_pool), 2, replace=False)
            p1, p2 = pts_pool[idx]
            dx, dz = p2[0] - p1[0], p2[1] - p1[1]
            length = np.hypot(dx, dz)
            if length < 0.01: continue
            
            A, B = -dz / length, dx / length
            C = -(A * p1[0] + B * p1[1])
            
            dist = np.abs(A * pts_pool[:, 0] + B * pts_pool[:, 1] + C)
            inliers = np.where(dist < residual_threshold)[0]
            
            if len(inliers) > len(best_inliers):
                best_inliers = inliers
                
        if len(best_inliers) >= 20:
            clean_wall_points.append(pts_pool[best_inliers])
            pts_pool = np.delete(pts_pool, best_inliers, axis=0)
        else:
            break # No more dense lines found
            
    # Fallback if the data is completely sparse
    if not clean_wall_points:
        combined_clean_points = points_2d
    else:
        combined_clean_points = np.vstack(clean_wall_points)

    debug_info['ransac_point_count'] = int(len(combined_clean_points))
    print(f"RANSAC-clean point count: {len(combined_clean_points)}")

    if len(combined_clean_points) >= 3:
        clustering_eps = dbscan_eps if dbscan_eps is not None else max(resolution * 4, 0.12)
        clustering = DBSCAN(eps=clustering_eps, min_samples=dbscan_min_samples)
        cluster_labels = clustering.fit_predict(combined_clean_points)
        valid_mask = cluster_labels != -1
        valid_points = combined_clean_points[valid_mask]
        unique_labels, counts = np.unique(cluster_labels[valid_mask], return_counts=True) if np.any(valid_mask) else (np.array([]), np.array([]))
        noise_count = int(np.sum(cluster_labels == -1))
        print(f"DBSCAN clusters found: {len(unique_labels)} (noise points: {noise_count})")
        debug_info['dbscan_cluster_count'] = int(len(unique_labels))
        debug_info['dbscan_noise_count'] = noise_count

        if len(unique_labels) > 0:
            cluster_sizes = {label: count for label, count in zip(unique_labels.tolist(), counts.tolist())}
            largest_cluster_size = max(cluster_sizes.values())
            min_cluster_size = max(3, int(np.ceil(largest_cluster_size * cluster_keep_ratio)))
            ordered_labels = sorted(cluster_sizes.keys(), key=lambda label: cluster_sizes[label], reverse=True)

            kept_clusters = []
            for label in ordered_labels:
                if cluster_sizes[label] < min_cluster_size:
                    continue
                kept_clusters.append(combined_clean_points[cluster_labels == label])
                if len(kept_clusters) >= max_clusters_to_keep:
                    break

            if kept_clusters:
                combined_clean_points = np.vstack(kept_clusters)
            elif len(valid_points) >= 3:
                combined_clean_points = valid_points
            else:
                print("DBSCAN removed too many points; falling back to RANSAC-clean points.")
        else:
            print("DBSCAN found only noise; falling back to RANSAC-clean points.")
    else:
        print("DBSCAN skipped due to insufficient points; falling back to unclustered clean points.")

    debug_info['kept_point_count'] = int(len(combined_clean_points))

    if len(combined_clean_points) < 3:
        print("Not enough clustered points for minAreaRect; returning empty mesh.")
        if return_debug:
            return np.empty((0, 3)), debug_info
        return np.empty((0, 3))

    # --- 2. GLOBAL BALANCED ORTHOGONAL FIT ---
    # Finds the optimal rectangle balancing all clean wall points
    rect = cv2.minAreaRect(combined_clean_points)
    rect_width, rect_depth = rect[1]
    debug_info['rect_width'] = float(rect_width)
    debug_info['rect_depth'] = float(rect_depth)
    print(f"minAreaRect dimensions: width={rect_width:.3f}, depth={rect_depth:.3f}")
    box = cv2.boxPoints(rect)
    
    # --- 3. EXTRUDE MESH ---
    y_min = trajectory_df['translation'].apply(lambda x: x[1]).min() - 0.2
    y_max = trajectory_df['translation'].apply(lambda x: x[1]).max() + 1.5
    height_grid = np.arange(y_min, y_max, resolution)
    
    dense_points = []
    for i in range(4):
        p1 = box[i]
        p2 = box[(i + 1) % 4]
        
        dist = np.linalg.norm(p2 - p1)
        num_steps = max(2, int(dist / resolution))
        
        x_line = np.linspace(p1[0], p2[0], num_steps)
        z_line = np.linspace(p1[1], p2[1], num_steps)
        
        for h in height_grid:
            dense_points.append(np.column_stack([x_line, np.full_like(x_line, h), z_line]))
            
    dense_mesh = np.vstack(dense_points) if dense_points else np.empty((0, 3))
    if return_debug:
        return dense_mesh, debug_info
    return dense_mesh

def project_textures_to_mesh(mesh_points, video_path, pose_df, K, video_clap_frame, max_keyframes=45):
    cap = cv2.VideoCapture(video_path)
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    
    point_colors = np.ones((len(mesh_points), 3)) * 0.15 # Fallback dark gray
    best_depth = np.full(len(mesh_points), np.inf) # Z-Buffer for Winner-Takes-All
    
    times = pose_df['timestamp'].values
    t_min, t_max = times.min(), times.max()
    
    # 1. Collect all valid frames
    candidates, frame_idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        t_query = (frame_idx - video_clap_frame) / video_fps
        if t_min <= t_query <= t_max:
            if frame_idx % 10 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                candidates.append((frame_idx, np.var(gray), t_query))
        frame_idx += 1
        
    # 2. CRITICAL FIX: Temporal NMS Keyframe Selection
    candidates.sort(key=lambda x: x[1], reverse=True)
    keyframes = []
    
    for cand in candidates:
        # Enforce a strict 1-second minimum gap between selected frames
        # This creates massive, solid patches and eliminates jagged micro-seams
        is_distinct = True
        for kf in keyframes:
            if abs(cand[2] - kf[2]) < 1: 
                is_distinct = False
                break
                
        if is_distinct:
            keyframes.append(cand)
            
        if len(keyframes) >= max_keyframes:
            break
            
    translations = np.array(pose_df['translation'].tolist())
    rotations = np.array(pose_df['rotation_matrix'].tolist())
    
    # 3. Strict Winner-Takes-All Projection (No Blending = No Ghosting)
    for f_idx, _, t_query in keyframes:
        idx = np.clip(np.searchsorted(times, t_query) - 1, 0, len(times) - 2)
        weight_t = (t_query - times[idx]) / (times[idx+1] - times[idx])
        
        t_w = (1 - weight_t) * translations[idx] + weight_t * translations[idx+1]
        slerp = Slerp(times[idx:idx+2], Rotation.from_matrix(rotations[idx:idx+2]))
        R_cw = slerp([t_query]).as_matrix()[0]
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
        ret, frame = cap.read()
        if not ret: continue
            
        X_centered = mesh_points - t_w
        X_cam = X_centered @ R_cw 
        
        X_opencv = np.zeros_like(X_cam)
        X_opencv[:, 0] =  X_cam[:, 0]  
        X_opencv[:, 1] = -X_cam[:, 1]  
        X_opencv[:, 2] = -X_cam[:, 2]  
        
        valid_depth = X_opencv[:, 2] > 0.1
        valid_idx = np.where(valid_depth)[0]
        if len(valid_idx) == 0: continue
            
        X_v = X_opencv[valid_idx]
        u = (X_v[:, 0] * K[0,0] / X_v[:, 2]) + K[0,2]
        v = (X_v[:, 1] * K[1,1] / X_v[:, 2]) + K[1,2]
        
        in_frame = (u >= 0) & (u < width - 1) & (v >= 0) & (v < height - 1)
        in_frame_idx = valid_idx[in_frame]
        if len(in_frame_idx) == 0: continue
            
        current_depths = X_v[in_frame, 2]
        better_view_mask = current_depths < best_depth[in_frame_idx]
        
        if not np.any(better_view_mask): continue
            
        update_idx = in_frame_idx[better_view_mask]
        u_win = u[in_frame][better_view_mask]
        v_win = v[in_frame][better_view_mask]
        
        u0, v0 = np.floor(u_win).astype(int), np.floor(v_win).astype(int)
        u1, v1 = u0 + 1, v0 + 1
        wu, wv = u_win - u0, v_win - v0
        
        sampled = (
            frame[v0, u0] * (1 - wu)[:, None] * (1 - wv)[:, None] +
            frame[v0, u1] * wu[:, None] * (1 - wv)[:, None] +
            frame[v1, u0] * (1 - wu)[:, None] * wv[:, None] +
            frame[v1, u1] * wu[:, None] * wv[:, None]
        )
        
        # Pure overwrite. 
        point_colors[update_idx] = sampled[:, ::-1] / 255.0
        best_depth[update_idx] = current_depths[better_view_mask]
        
    cap.release()
    point_colors = np.clip(point_colors, 0.0, 1.0)
    
    colors_uint8 = (point_colors * 255).astype(np.uint8)
    plotly_colors = [f'rgb({c[0]}, {c[1]}, {c[2]})' for c in colors_uint8]
    
    return plotly_colors


# =====================================================================
# 5. VISUALIZATION
# =====================================================================

def display_reconstruction_results(mesh_points, mesh_colors, hough_points, trajectory_df):
    trajectory = np.array(trajectory_df['translation'].tolist())
    
    # ---------------------------------------------------------
    # PLOT 1: THE BEAUTY PASS (Clean Textured Room)
    # ---------------------------------------------------------
    trace_clean_mesh = go.Scatter3d(
        x=mesh_points[:, 0], y=mesh_points[:, 1], z=mesh_points[:, 2],
        mode='markers',
        marker=dict(size=4, color=mesh_colors, opacity=1.0),
        name='Textured Room Mesh'
    )
    
    layout_clean = go.Layout(
        title="Final Output: Clean Textured Room Reconstruction",
        scene=dict(
            xaxis=dict(title='X (m)', backgroundcolor='black', gridcolor='#333333'),
            yaxis=dict(title='Y (m)', backgroundcolor='black', gridcolor='#333333'),
            zaxis=dict(title='Z (m)', backgroundcolor='black', gridcolor='#333333'),
            bgcolor='black', aspectmode='data'
        ),
        paper_bgcolor='black', font=dict(color='white'), margin=dict(l=0, r=0, t=40, b=0)
    )
    go.Figure(data=[trace_clean_mesh], layout=layout_clean).show()

    # ---------------------------------------------------------
    # PLOT 2: THE DIAGNOSTIC PASS (Geometry vs Raw Data)
    # ---------------------------------------------------------
    traces_diagnostic = [
        go.Scatter3d(
            x=mesh_points[:, 0], y=mesh_points[:, 1], z=mesh_points[:, 2],
            mode='markers',
            marker=dict(size=4, color=mesh_colors, opacity=0.9), # Slightly transparent
            name='Textured Room Mesh'
        ),
        go.Scatter3d(
            x=hough_points[:, 0], y=hough_points[:, 1], z=hough_points[:, 2],
            mode='markers',
            marker=dict(size=2, color='rgba(0, 255, 255, 0.3)'), # Ghosted cyan
            name='Raw Acoustic Voxels'
        ),
        go.Scatter3d(
            x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
            mode='lines+markers',
            line=dict(color='yellow', width=4),
            marker=dict(size=2, color='white'),
            name='Camera Path'
        )
    ]
    
    layout_diagnostic = go.Layout(
        title="Diagnostic View: Geometry vs. Raw Acoustic Data",
        scene=dict(
            xaxis=dict(title='X (m)', backgroundcolor='black', gridcolor='#333333'),
            yaxis=dict(title='Y (m)', backgroundcolor='black', gridcolor='#333333'),
            zaxis=dict(title='Z (m)', backgroundcolor='black', gridcolor='#333333'),
            bgcolor='black', aspectmode='data'
        ),
        paper_bgcolor='black', font=dict(color='white'), margin=dict(l=0, r=0, t=40, b=0)
    )
    go.Figure(data=traces_diagnostic, layout=layout_diagnostic).show()

# =====================================================================
# MASTER EXECUTION 
# =====================================================================
if __name__ == "__main__":
    print("Initializing Active Room Reconstruction Engine...")
    
    VIDEO_FPS = 60.0

    print("Parsing telemetry...")
    trajectory_df = parse_blender_tracking_log(py_path, fps=VIDEO_FPS)
    K, video_width, video_height = parse_blender_intrinsics(py_path)
    sr, audio_sig = wavfile.read(wav_path)

    # Reference signal
    t_chirp = np.linspace(0, 0.010, int(44100 * 0.010), endpoint=False)
    tapered_chirp = np.sin(2 * np.pi * (18000 + (22000 - 18000) * t_chirp / (2 * 0.010))) * signal.windows.tukey(len(t_chirp), alpha=0.1)

    # Align Timelines
    trajectory_df['timestamp'] = trajectory_df['timestamp'] - (VIDEO_CLAP_FRAME / VIDEO_FPS)

    print("Extracting acoustic ToF distances (SNR filtered)...")
    acoustic_distances = compute_acoustic_distances(audio_sig, trajectory_df, sr, tapered_chirp)
    
    print("Accumulating 3D Hough Space...")
    voxel_grid, bounds, v_scale = build_hough_accumulator(trajectory_df, acoustic_distances, voxel_size=0.04)

    print("Extracting high-confidence acoustic voxels...")
    voxel_threshold, nonzero_count, selected_percentile = select_adaptive_voxel_threshold(
        voxel_grid,
        dense_percentile=95,
        sparse_percentile=90,
        density_threshold=2000,
    )

    if voxel_threshold is None:
        hough_points = np.empty((0, 3))
    else:
        voxel_indices = np.argwhere(voxel_grid >= voxel_threshold)
        hough_points = bounds + (voxel_indices * v_scale)
        print(f"Adaptive voxel extraction produced {len(hough_points)} candidate points.")

    # CRITICAL FIX: Calling the new RANSAC function instead of the old Rectangle function
    print("Calculating Iterative RANSAC Planes...")
    ideal_room_mesh = generate_balanced_room_mesh(hough_points, trajectory_df, residual_threshold=0.08, resolution=0.03)

    print(f"Projecting textures onto {len(ideal_room_mesh)} mesh points...")
    mesh_colors = project_textures_to_mesh(ideal_room_mesh, mov_path, trajectory_df, K, VIDEO_CLAP_FRAME)

    print("Launching final geometry scenes...")
    display_reconstruction_results(ideal_room_mesh, mesh_colors, hough_points, trajectory_df)
    print("Execution complete!")

Initializing Active Room Reconstruction Engine...
Parsing telemetry...
Extracting acoustic ToF distances (SNR filtered)...


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/3996235030.py:479: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_path)


Accumulating 3D Hough Space...
Extracting high-confidence acoustic voxels...
Adaptive voxel threshold: 95th percentile over 157120 non-zero voxels -> 11.000
Adaptive voxel extraction produced 9174 candidate points.
Calculating Iterative RANSAC Planes...
generate_balanced_room_mesh input point count: 9174
RANSAC-clean point count: 7750
DBSCAN clusters found: 12 (noise points: 2)
minAreaRect dimensions: width=4.598, depth=3.075
Projecting textures onto 33660 mesh points...
Launching final geometry scenes...


Execution complete!


In [41]:
import os
import pandas as pd


def compute_camera_path_bounds(trajectory_df):
    trajectory = np.array(trajectory_df['translation'].tolist())
    x_min, y_min, z_min = trajectory.min(axis=0)
    x_max, y_max, z_max = trajectory.max(axis=0)
    camera_width = float(x_max - x_min)
    camera_depth = float(z_max - z_min)
    camera_area = float(camera_width * camera_depth)
    return {
        'x_min': float(x_min),
        'x_max': float(x_max),
        'z_min': float(z_min),
        'z_max': float(z_max),
        'camera_width': camera_width,
        'camera_depth': camera_depth,
        'camera_area': camera_area,
    }


def find_dataset_files(data_folder):
    mov_file = None
    py_file = None
    wav_file = None
    frame_file = os.path.join(data_folder, 'frame.txt')

    for filename in os.listdir(data_folder):
        if filename.endswith('.mov'):
            mov_file = os.path.join(data_folder, filename)
        elif filename.endswith('.blender.py'):
            py_file = os.path.join(data_folder, filename)
        elif filename.endswith('.wav'):
            wav_file = os.path.join(data_folder, filename)

    if mov_file is None or py_file is None:
        raise FileNotFoundError(f'Missing .mov or .blender.py in {data_folder}')

    if wav_file is None:
        wav_file = os.path.join(data_folder, os.path.basename(mov_file).replace('.mov', '.wav'))
        if not os.path.exists(wav_file):
            extract_audio_from_mov(mov_file, wav_file)

    video_clap_frame = int(open(frame_file).read().strip())
    return mov_file, wav_file, py_file, video_clap_frame


def run_reconstruction_diagnostics(data_folder, video_fps=60.0, voxel_size=None):
    voxel_size = voxel_size if voxel_size is not None else globals().get('HOUGH_VOXEL_SIZE', 0.04)
    mov_file, wav_file, py_file, video_clap_frame = find_dataset_files(data_folder)

    trajectory_df = parse_blender_tracking_log(py_file, fps=video_fps)
    K, _, _ = parse_blender_intrinsics(py_file)
    sr, audio_sig = wavfile.read(wav_file)

    t_chirp = np.linspace(0, 0.010, int(44100 * 0.010), endpoint=False)
    tapered_chirp = np.sin(2 * np.pi * (18000 + (22000 - 18000) * t_chirp / (2 * 0.010))) * signal.windows.tukey(len(t_chirp), alpha=0.1)

    trajectory_df['timestamp'] = trajectory_df['timestamp'] - (video_clap_frame / video_fps)

    acoustic_distances = compute_acoustic_distances(audio_sig, trajectory_df, sr, tapered_chirp)
    # new acoustic diagnostic: count valid (non-NaN) distance measurements
    valid_dist_count = int(np.sum(~np.isnan(acoustic_distances)))

    voxel_grid, bounds, v_scale = build_hough_accumulator(trajectory_df, acoustic_distances, voxel_size=voxel_size)
    voxel_threshold, nonzero_count, selected_percentile = select_adaptive_voxel_threshold(
        voxel_grid,
        dense_percentile=95,
        sparse_percentile=90,
        density_threshold=2000,
    )

    if voxel_threshold is None:
        hough_points = np.empty((0, 3))
    else:
        voxel_indices = np.argwhere(voxel_grid >= voxel_threshold)
        hough_points = bounds + (voxel_indices * v_scale)

    camera_bounds = compute_camera_path_bounds(trajectory_df)

    # --- CAMERA-PATH ORIENTED RECTANGLE (ground-truth proxy) ---
    traj = np.array(trajectory_df['translation'].tolist())
    traj_2d = np.float32(traj[:, [0, 2]])
    if len(traj_2d) >= 3:
        cam_rect = cv2.minAreaRect(traj_2d)
        cam_rect_width, cam_rect_depth = cam_rect[1]
        cam_box = cv2.boxPoints(cam_rect)
        cam_rect_center = (float(cam_rect[0][0]), float(cam_rect[0][1]))
        cam_rect_area = float(cam_rect_width * cam_rect_depth)
        cam_rect_angle = float(cam_rect[2])
    else:
        cam_rect_width = cam_rect_depth = cam_rect_area = cam_rect_angle = np.nan
        cam_box = np.empty((0, 2))
        cam_rect_center = (np.nan, np.nan)

    ideal_room_mesh, room_debug = generate_balanced_room_mesh(
        hough_points,
        trajectory_df,
        residual_threshold=0.08,
        resolution=0.03,
        return_debug=True,
    )

    fitted_area = float(room_debug['rect_width'] * room_debug['rect_depth']) if room_debug['rect_width'] > 0 and room_debug['rect_depth'] > 0 else 0.0
    fit_to_camera_ratio = fitted_area / camera_bounds['camera_area'] if camera_bounds['camera_area'] > 0 else np.nan
    fit_to_camera_rect_ratio = fitted_area / cam_rect_area if cam_rect_area > 0 else np.nan

    if fit_to_camera_ratio > 1.35:
        constraint_hint = 'Room footprint is likely too large; tighten DBSCAN or raise the voxel threshold slightly.'
    elif fit_to_camera_ratio < 0.75:
        constraint_hint = 'Room footprint is likely too small; loosen DBSCAN or lower the voxel threshold slightly.'
    else:
        constraint_hint = 'Room footprint is in a reasonable range relative to the camera-path bounds.'

    # Additional derived diagnostics
    camera_area = camera_bounds['camera_area']
    supports_per_camera_area = valid_dist_count / camera_area if camera_area > 0 else np.nan
    points_per_support = int(len(hough_points) / valid_dist_count) if valid_dist_count > 0 else np.nan
    nonzero_voxel_density = int(nonzero_count) / camera_area if camera_area > 0 else np.nan

    # --- Per-frame SNR / peak diagnostics (save CSV + PNG) ---
    per_frame_csv = None
    peak_png = None
    snr_png = None
    try:
        import matplotlib.pyplot as plt
        os.makedirs('diagnostics', exist_ok=True)
        window_samples = int(0.100 * sr)
        timestamps = trajectory_df['timestamp'].to_numpy()
        center_indices = (timestamps * sr).astype(int)
        rows = []
        for i, idx_center in enumerate(center_indices):
            if idx_center + window_samples > len(audio_sig):
                rows.append({'frame_idx': i, 'timestamp': float(timestamps[i]), 'center_idx': int(idx_center), 'peak_count': 0, 'max_peak': np.nan, 'snr': np.nan})
                continue
            audio_window = audio_sig[idx_center: idx_center + window_samples]
            if audio_window.ndim > 1:
                audio_window = audio_window[:, 0]
            if len(audio_window) < len(tapered_chirp):
                rows.append({'frame_idx': i, 'timestamp': float(timestamps[i]), 'center_idx': int(idx_center), 'peak_count': 0, 'max_peak': np.nan, 'snr': np.nan})
                continue
            xcorr = signal.correlate(audio_window, tapered_chirp, mode='valid')
            noise_floor = np.sqrt(np.mean(audio_window**2)) + 1e-6
            peaks, props = signal.find_peaks(xcorr, height=noise_floor * 1.0, distance=int(0.001 * sr))
            peak_heights = props['peak_heights'] if 'peak_heights' in props else None
            peak_count = len(peaks)
            max_peak = float(np.max(peak_heights)) if peak_count > 0 else np.nan
            snr_est = max_peak / noise_floor if peak_count > 0 else np.nan
            rows.append({'frame_idx': i, 'timestamp': float(timestamps[i]), 'center_idx': int(idx_center), 'peak_count': int(peak_count), 'max_peak': max_peak, 'snr': snr_est})

        per_df = pd.DataFrame(rows)
        base = os.path.basename(data_folder).replace('/', '_')
        per_frame_csv = f'diagnostics/{base}_per_frame_supports.csv'
        peak_png = f'diagnostics/{base}_peak_count.png'
        snr_png = f'diagnostics/{base}_snr.png'
        per_df.to_csv(per_frame_csv, index=False)

        plt.figure(figsize=(10, 3))
        plt.plot(per_df['timestamp'], per_df['peak_count'], marker='o')
        plt.title(f'{data_folder} - peak count per timestamp')
        plt.xlabel('time (s)')
        plt.ylabel('peak count')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(peak_png)
        plt.close()

        plt.figure(figsize=(10, 3))
        plt.plot(per_df['timestamp'], per_df['snr'], marker='o')
        plt.title(f'{data_folder} - estimated SNR per timestamp')
        plt.xlabel('time (s)')
        plt.ylabel('SNR (rel)')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(snr_png)
        plt.close()
    except Exception as e:
        print('Per-frame diagnostics failed:', e)

    return {
        'data_folder': data_folder,
        'mov_file': mov_file,
        'video_clap_frame': video_clap_frame,
        'nonzero_voxels': int(nonzero_count),
        'selected_percentile': int(selected_percentile),
        'voxel_threshold': float(voxel_threshold) if voxel_threshold is not None else np.nan,
        'hough_point_count': int(len(hough_points)),
        'valid_dist_count': valid_dist_count,
        'points_per_support': points_per_support,
        'supports_per_camera_area': float(supports_per_camera_area),
        'nonzero_voxel_density': float(nonzero_voxel_density),
        'camera_width': camera_bounds['camera_width'],
        'camera_depth': camera_bounds['camera_depth'],
        'camera_area': camera_area,
        'camera_rect_width': float(cam_rect_width) if not np.isnan(cam_rect_width) else np.nan,
        'camera_rect_depth': float(cam_rect_depth) if not np.isnan(cam_rect_depth) else np.nan,
        'camera_rect_area': float(cam_rect_area) if not np.isnan(cam_rect_area) else np.nan,
        'camera_rect_angle': float(cam_rect_angle) if not np.isnan(cam_rect_angle) else np.nan,
        'fit_to_camera_rect_ratio': fit_to_camera_rect_ratio,
        'per_frame_csv': per_frame_csv,
        'peak_png': peak_png,
        'snr_png': snr_png,
        'ransac_point_count': int(room_debug['ransac_point_count']),
        'dbscan_cluster_count': int(room_debug['dbscan_cluster_count']),
        'dbscan_noise_count': int(room_debug['dbscan_noise_count']),
        'kept_point_count': int(room_debug['kept_point_count']),
        'rect_width': float(room_debug['rect_width']),
        'rect_depth': float(room_debug['rect_depth']),
        'rect_area': fitted_area,
        'fit_to_camera_ratio': fit_to_camera_ratio,
        'constraint_hint': constraint_hint,
    }


compare_folders = ['Data/TestData1', 'Data/TestData2']
comparison_rows = []
voxel_size = globals().get('HOUGH_VOXEL_SIZE', 0.04)
for folder in compare_folders:
    print(f'\n--- Diagnostic run for {folder} ---')
    result = run_reconstruction_diagnostics(folder, video_fps=VIDEO_FPS, voxel_size=voxel_size)
    comparison_rows.append(result)
    print(
        f"camera bounds={result['camera_width']:.3f}m x {result['camera_depth']:.3f}m, "
        f"camera rect={result['camera_rect_width']:.3f}m x {result['camera_rect_depth']:.3f}m, "
        f"rect={result['rect_width']:.3f}m x {result['rect_depth']:.3f}m, "
        f"ratio={result['fit_to_camera_ratio']:.3f}, valid_supports={result['valid_dist_count']}"
    )
    print(result['constraint_hint'])
    print('Per-frame CSV:', result.get('per_frame_csv'))
    print('Peak PNG:', result.get('peak_png'))
    print('SNR PNG:', result.get('snr_png'))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df[[
    'data_folder', 'nonzero_voxels', 'selected_percentile', 'valid_dist_count', 'points_per_support',
    'supports_per_camera_area', 'nonzero_voxel_density', 'hough_point_count', 'camera_width', 'camera_depth',
    'camera_rect_width', 'camera_rect_depth', 'camera_rect_area', 'fit_to_camera_rect_ratio',
    'per_frame_csv', 'peak_png', 'snr_png',
    'rect_width', 'rect_depth', 'fit_to_camera_ratio', 'dbscan_cluster_count', 'dbscan_noise_count', 'constraint_hint'
]]

# --- Overlay visualization: camera rect, fitted rect, and Hough points ---

import matplotlib.pyplot as plt
import numpy as np
os.makedirs('diagnostics', exist_ok=True)
compare_folders = ['Data/TestData1', 'Data/TestData2']
voxel_size = globals().get('HOUGH_VOXEL_SIZE', 0.04)
for folder in compare_folders:
    try:
        mov_file, wav_file, py_file, video_clap_frame = find_dataset_files(folder)
        traj = parse_blender_tracking_log(py_file, fps=VIDEO_FPS)
        traj['timestamp'] = traj['timestamp'] - (video_clap_frame / VIDEO_FPS)
        sr, audio_sig = wavfile.read(wav_file)
        t_chirp = np.linspace(0, 0.010, int(44100 * 0.010), endpoint=False)
        tapered_chirp = np.sin(2 * np.pi * (18000 + (22000 - 18000) * t_chirp / (2 * 0.010))) * signal.windows.tukey(len(t_chirp), alpha=0.1)
        acoustic_distances = compute_acoustic_distances(audio_sig, traj, sr, tapered_chirp)
        voxel_grid, bounds, v_scale = build_hough_accumulator(traj, acoustic_distances, voxel_size=voxel_size)
        voxel_threshold, nonzero_count, selected_percentile = select_adaptive_voxel_threshold(voxel_grid)
        if voxel_threshold is None:
            print(f'No Hough points for {folder}')
            continue
        voxel_indices = np.argwhere(voxel_grid >= voxel_threshold)
        hough_points = bounds + (voxel_indices * v_scale)
        # Camera rect (X,Z plane)
        traj_arr = np.array(traj['translation'].tolist())
        traj_2d = np.float32(traj_arr[:, [0, 2]])
        if len(traj_2d) >= 3:
            cam_rect = cv2.minAreaRect(traj_2d)
            cam_box = cv2.boxPoints(cam_rect)
        else:
            cam_box = np.empty((0, 2))
        # Fitted rect from Hough points (approximate; uses all Hough points)
        if len(hough_points) >= 3:
            hough_2d = np.float32(hough_points[:, [0, 2]])
            fit_rect = cv2.minAreaRect(hough_2d)
            fit_box = cv2.boxPoints(fit_rect)
        else:
            fit_box = np.empty((0, 2))
        plt.figure(figsize=(6, 6))
        if len(hough_points) > 0:
            plt.scatter(hough_points[:, 0], hough_points[:, 2], s=2, c='cyan', alpha=0.6, label='Hough points')
        if traj_2d.size:
            plt.plot(traj_2d[:, 0], traj_2d[:, 1], '-k', lw=1, label='Camera path')
        if cam_box.size:
            cam_box_closed = np.vstack([cam_box, cam_box[0]])
            plt.plot(cam_box_closed[:, 0], cam_box_closed[:, 1], '-r', lw=2, label='Camera rect')
        if fit_box.size:
            fit_box_closed = np.vstack([fit_box, fit_box[0]])
            plt.plot(fit_box_closed[:, 0], fit_box_closed[:, 1], '-g', lw=2, label='Fitted rect (Hough)')
        plt.gca().set_aspect('equal', 'box')
        plt.title(f'{folder} overlay (X vs Z)')
        plt.legend(loc='upper right')
        out_path = f'diagnostics/{os.path.basename(folder)}_overlay.png'
        plt.tight_layout()
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()
        print('Saved overlay:', out_path)
    except Exception as e:
        print('Overlay failed for', folder, e)

# --- Automated parameter sweep (DBSCAN / voxel percentile / RANSAC) ---
def run_parameter_sweep(compare_folders, voxel_size=0.04):
    import itertools, time, csv
    os.makedirs('diagnostics', exist_ok=True)
    results = []
    # parameter grid (kept small to finish quickly)
    dbscan_eps_list = [0.12, 0.18, 0.24]
    cluster_keep_ratio_list = [0.15, 0.25, 0.35]
    sparse_percentile_list = [85, 90]
    residual_list = [0.08, 0.10]
    for folder in compare_folders:
        mov_file, wav_file, py_file, video_clap_frame = find_dataset_files(folder)
        traj = parse_blender_tracking_log(py_file, fps=VIDEO_FPS)
        traj['timestamp'] = traj['timestamp'] - (video_clap_frame / VIDEO_FPS)
        camera_bounds = compute_camera_path_bounds(traj)
        # build base acoustic/hough once per folder (we'll rethreshold)
        sr, audio_sig = wavfile.read(wav_file)
        t_chirp = np.linspace(0, 0.010, int(44100 * 0.010), endpoint=False)
        tapered_chirp = np.sin(2 * np.pi * (18000 + (22000 - 18000) * t_chirp / (2 * 0.010))) * signal.windows.tukey(len(t_chirp), alpha=0.1)
        acoustic_distances = compute_acoustic_distances(audio_sig, traj, sr, tapered_chirp)
        voxel_grid, bounds, v_scale = build_hough_accumulator(traj, acoustic_distances, voxel_size=voxel_size)
        nonzero_voxels = voxel_grid[voxel_grid > 0]
        for db_eps, keep_ratio, sparse_pct, resid in itertools.product(dbscan_eps_list, cluster_keep_ratio_list, sparse_percentile_list, residual_list):
            t0 = time.time()
            # choose threshold by percentile
            if len(nonzero_voxels)==0:
                continue
            thresh = np.percentile(nonzero_voxels, sparse_pct)
            voxel_indices = np.argwhere(voxel_grid >= thresh)
            hough_points = bounds + (voxel_indices * v_scale) if len(voxel_indices)>0 else np.empty((0,3))
            # fit
            if len(hough_points) < 3:
                fit_area = 0.0
                fit_ratio = 0.0
                coverage = 0.0
            else:
                # call generate_balanced_room_mesh to get debug (pass dbscan eps and cluster_keep_ratio)
                mesh, dbg = generate_balanced_room_mesh(hough_points, traj, residual_threshold=resid, resolution=0.03, dbscan_eps=db_eps, cluster_keep_ratio=keep_ratio, return_debug=True)
                fit_area = float(dbg['rect_width'] * dbg['rect_depth']) if dbg['rect_width']>0 and dbg['rect_depth']>0 else 0.0
                cam_area = camera_bounds['camera_area'] if camera_bounds['camera_area']>0 else 1.0
                fit_ratio = fit_area / cam_area
                # perimeter coverage: sample points along fitted rectangle edges and check proximity to hough_points
                try:
                    rect = cv2.minAreaRect(np.float32(hough_points[:, [0,2]]))
                    box = cv2.boxPoints(rect)
                    edge_pts = []
                    for i in range(4):
                        p1 = box[i]; p2 = box[(i+1)%4]
                        for s in np.linspace(0,1,25):
                            edge_pts.append(p1*(1-s)+p2*s)
                    edge_pts = np.array(edge_pts)
                    if len(edge_pts)>0 and len(hough_points)>0:
                        dists = np.sqrt(((edge_pts[:,None,0]-hough_points[None,:,0])**2)+((edge_pts[:,None,1]-hough_points[None,:,2])**2))
                        close = np.any(dists < 0.18, axis=1)
                        coverage = float(np.sum(close))/len(close)
                    else:
                        coverage = 0.0
                except Exception:
                    coverage = 0.0
            score = coverage - 0.5*abs(1.0 - fit_ratio)
            results.append({'folder': folder, 'dbscan_eps': db_eps, 'cluster_keep_ratio': keep_ratio, 'sparse_percentile': sparse_pct, 'residual': resid, 'fit_ratio': fit_ratio, 'coverage': coverage, 'score': score, 'hough_count': int(len(hough_points))})
    # save results
    out_csv = 'diagnostics/parameter_sweep_results.csv'
    pd.DataFrame(results).to_csv(out_csv, index=False)
    print('Saved parameter sweep results ->', out_csv)
    # pick best per folder and save overlay for that config
    df = pd.DataFrame(results)
    for folder in df['folder'].unique():
        sub = df[df['folder']==folder].sort_values('score', ascending=False).head(1).iloc[0]
        print('Best for', folder, ':', dict(sub))
        # re-generate overlay for best config
        try:
            mov_file, wav_file, py_file, video_clap_frame = find_dataset_files(folder)
            traj = parse_blender_tracking_log(py_file, fps=VIDEO_FPS)
            traj['timestamp'] = traj['timestamp'] - (video_clap_frame / VIDEO_FPS)
            sr, audio_sig = wavfile.read(wav_file)
            acoustic_distances = compute_acoustic_distances(audio_sig, traj, sr, tapered_chirp)
            voxel_grid, bounds, v_scale = build_hough_accumulator(traj, acoustic_distances, voxel_size=voxel_size)
            thresh = np.percentile(voxel_grid[voxel_grid>0], int(sub['sparse_percentile']))
            voxel_indices = np.argwhere(voxel_grid >= thresh)
            hough_points = bounds + (voxel_indices * v_scale)
            mesh, dbg = generate_balanced_room_mesh(hough_points, traj, residual_threshold=float(sub['residual']), resolution=0.03, dbscan_eps=float(sub['dbscan_eps']), cluster_keep_ratio=float(sub['cluster_keep_ratio']), return_debug=True)
            # plot overlay
            plt.figure(figsize=(6,6))
            if len(hough_points)>0: plt.scatter(hough_points[:,0], hough_points[:,2], s=2, c='cyan', alpha=0.6)
            traj_arr = np.array(traj['translation'].tolist()); traj_2d = traj_arr[:,[0,2]]
            plt.plot(traj_2d[:,0], traj_2d[:,1], '-k', lw=1)
            cam_rect = cv2.minAreaRect(np.float32(traj_2d)); cam_box = cv2.boxPoints(cam_rect); cam_box_closed = np.vstack([cam_box, cam_box[0]]); plt.plot(cam_box_closed[:,0], cam_box_closed[:,1], '-r', lw=2)
            try:
                rect = cv2.minAreaRect(np.float32(hough_points[:, [0,2]])); fit_box = cv2.boxPoints(rect); fit_box_closed = np.vstack([fit_box, fit_box[0]]); plt.plot(fit_box_closed[:,0], fit_box_closed[:,1], '-g', lw=2)
            except Exception:
                pass
            plt.gca().set_aspect('equal','box')
            out = f'diagnostics/{os.path.basename(folder)}_best_config_overlay.png'
            plt.title(f'{folder} best-config overlay')
            plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
            print('Saved best-config overlay ->', out)
        except Exception as e:
            print('Failed to save best overlay for', folder, e)
    return df

# Run sweep (small grid) now
sweep_df = run_parameter_sweep(['Data/TestData1','Data/TestData2'], voxel_size=voxel_size)
print('Parameter sweep completed. Top rows:')
print(sweep_df.sort_values('score', ascending=False).head())


--- Diagnostic run for Data/TestData1 ---


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/2106445183.py:55: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_file)


Adaptive voxel threshold: 95th percentile over 157120 non-zero voxels -> 11.000
generate_balanced_room_mesh input point count: 9174
RANSAC-clean point count: 7765
DBSCAN clusters found: 13 (noise points: 0)
minAreaRect dimensions: width=4.490, depth=3.260
camera bounds=2.919m x 4.128m, camera rect=3.847m x 2.138m, rect=4.490m x 3.260m, ratio=1.215, valid_supports=1530
Room footprint is in a reasonable range relative to the camera-path bounds.
Per-frame CSV: diagnostics/TestData1_per_frame_supports.csv
Peak PNG: diagnostics/TestData1_peak_count.png
SNR PNG: diagnostics/TestData1_snr.png

--- Diagnostic run for Data/TestData2 ---


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/2106445183.py:55: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_file)


Adaptive voxel threshold: 95th percentile over 265923 non-zero voxels -> 18.000
generate_balanced_room_mesh input point count: 15046
RANSAC-clean point count: 12380
DBSCAN clusters found: 10 (noise points: 3)
minAreaRect dimensions: width=5.451, depth=1.946
camera bounds=6.804m x 4.974m, camera rect=7.469m x 3.922m, rect=5.451m x 1.946m, ratio=0.314, valid_supports=2494
Room footprint is likely too small; loosen DBSCAN or lower the voxel threshold slightly.
Per-frame CSV: diagnostics/TestData2_per_frame_supports.csv
Peak PNG: diagnostics/TestData2_peak_count.png
SNR PNG: diagnostics/TestData2_snr.png


...

generate_balanced_room_mesh input point count: 40203
RANSAC-clean point count: 27968
DBSCAN clusters found: 7 (noise points: 0)
minAreaRect dimensions: width=5.625, depth=3.507
generate_balanced_room_mesh input point count: 40203
RANSAC-clean point count: 30884
DBSCAN clusters found: 6 (noise points: 2)
minAreaRect dimensions: width=5.673, depth=3.501
generate_balanced_room_mesh input point count: 28833
RANSAC-clean point count: 21296
DBSCAN clusters found: 7 (noise points: 4)
minAreaRect dimensions: width=5.873, depth=3.002
generate_balanced_room_mesh input point count: 28833
RANSAC-clean point count: 23750
DBSCAN clusters found: 10 (noise points: 3)
minAreaRect dimensions: width=5.849, depth=2.952
generate_balanced_room_mesh input point count: 40203
RANSAC-clean point count: 27733
DBSCAN clusters found: 6 (noise points: 0)
minAreaRect dimensions: width=5.752, depth=3.532
generate_balanced_room_mesh input point count: 40203
RANSAC-clean point count: 30404
DBSCAN clusters found: 8 (noi

/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/2106445183.py:378: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_file)


generate_balanced_room_mesh input point count: 18700
RANSAC-clean point count: 15512
DBSCAN clusters found: 4 (noise points: 0)
minAreaRect dimensions: width=4.800, depth=3.708
Saved best-config overlay -> diagnostics/TestData1_best_config_overlay.png
Best for Data/TestData2 : {'folder': 'Data/TestData2', 'dbscan_eps': np.float64(0.24), 'cluster_keep_ratio': np.float64(0.35), 'sparse_percentile': np.int64(85), 'residual': np.float64(0.1), 'fit_ratio': np.float64(0.623798420185999), 'coverage': np.float64(0.22), 'score': np.float64(0.03189921009299948), 'hough_count': np.int64(40203)}


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/2106445183.py:378: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_file)


generate_balanced_room_mesh input point count: 40203
RANSAC-clean point count: 31081
DBSCAN clusters found: 3 (noise points: 0)
minAreaRect dimensions: width=5.740, depth=3.598
Saved best-config overlay -> diagnostics/TestData2_best_config_overlay.png
Parameter sweep completed. Top rows:
            folder  dbscan_eps  cluster_keep_ratio  sparse_percentile  \
23  Data/TestData1        0.18                0.35                 90   
10  Data/TestData1        0.12                0.35                 90   
2   Data/TestData1        0.12                0.15                 90   
18  Data/TestData1        0.18                0.25                 90   
26  Data/TestData1        0.24                0.15                 90   

    residual  fit_ratio  coverage     score  hough_count  
23      0.10   1.285617      0.38  0.237191        18700  
10      0.08   1.290403      0.38  0.234799        18700  
2       0.08   1.345886      0.38  0.207057        18700  
18      0.08   1.346111      0.38  0

In [43]:
# --- Auto-tune: save best params per-dataset and apply them ---
import json

def auto_tune_and_apply(sweep_csv='diagnostics/parameter_sweep_results.csv', voxel_size=None):
    import numpy as _np
    import json as _json
    import matplotlib.pyplot as _plt

    voxel_size = voxel_size if voxel_size is not None else globals().get('HOUGH_VOXEL_SIZE', 0.04)
    if not os.path.exists(sweep_csv):
        raise FileNotFoundError(f'Parameter sweep results not found: {sweep_csv}')

    df = pd.read_csv(sweep_csv)
    os.makedirs('diagnostics', exist_ok=True)

    summary = []
    video_fps = globals().get('VIDEO_FPS', 60.0)

    for folder in df['folder'].unique():
        sub = df[df['folder']==folder].sort_values('score', ascending=False).iloc[0]
        base = os.path.basename(folder)
        best_params = {
            'dbscan_eps': float(sub['dbscan_eps']),
            'cluster_keep_ratio': float(sub['cluster_keep_ratio']),
            'sparse_percentile': int(sub['sparse_percentile']),
            'residual': float(sub['residual'])
        }

        # Persist best params
        json_path = f'diagnostics/{base}_best_params.json'
        with open(json_path, 'w') as fh:
            _json.dump(best_params, fh, indent=2)
        print('Saved best params ->', json_path)

        # Re-run reconstruction using best params
        try:
            mov_file, wav_file, py_file, video_clap_frame = find_dataset_files(folder)
            traj = parse_blender_tracking_log(py_file, fps=video_fps)
            traj['timestamp'] = traj['timestamp'] - (video_clap_frame / video_fps)

            sr, audio_sig = wavfile.read(wav_file)
            t_chirp = np.linspace(0, 0.010, int(44100 * 0.010), endpoint=False)
            tapered_chirp = np.sin(2 * np.pi * (18000 + (22000 - 18000) * t_chirp / (2 * 0.010))) * signal.windows.tukey(len(t_chirp), alpha=0.1)

            acoustic_distances = compute_acoustic_distances(audio_sig, traj, sr, tapered_chirp)
            voxel_grid, bounds, v_scale = build_hough_accumulator(traj, acoustic_distances, voxel_size=voxel_size)
            nonzero = voxel_grid[voxel_grid > 0]
            if len(nonzero) == 0:
                print('No Hough voxels for', folder)
                continue

            thresh = float(_np.percentile(nonzero, best_params['sparse_percentile']))
            voxel_indices = np.argwhere(voxel_grid >= thresh)
            hough_points = bounds + (voxel_indices * v_scale) if len(voxel_indices) > 0 else _np.empty((0, 3))

            mesh, dbg = generate_balanced_room_mesh(
                hough_points,
                traj,
                residual_threshold=best_params['residual'],
                resolution=0.03,
                dbscan_eps=best_params['dbscan_eps'],
                cluster_keep_ratio=best_params['cluster_keep_ratio'],
                return_debug=True,
            )

            mesh_path = f'diagnostics/{base}_final_mesh.npy'
            _np.save(mesh_path, mesh)
            print('Saved mesh ->', mesh_path)

            # Overlay (Hough points, camera path, fitted rect)
            _plt.figure(figsize=(6, 6))
            if len(hough_points) > 0:
                _plt.scatter(hough_points[:, 0], hough_points[:, 2], s=2, c='cyan', alpha=0.6)
            traj_arr = _np.array(traj['translation'].tolist())
            traj_2d = traj_arr[:, [0, 2]]
            _plt.plot(traj_2d[:, 0], traj_2d[:, 1], '-k', lw=1)
            try:
                cam_rect = cv2.minAreaRect(_np.float32(traj_2d)); cam_box = cv2.boxPoints(cam_rect); cam_box_closed = _np.vstack([cam_box, cam_box[0]]); _plt.plot(cam_box_closed[:, 0], cam_box_closed[:, 1], '-r', lw=2)
            except Exception:
                pass
            try:
                if len(hough_points) >= 3:
                    rect = cv2.minAreaRect(_np.float32(hough_points[:, [0, 2]])); fit_box = cv2.boxPoints(rect); fit_box_closed = _np.vstack([fit_box, fit_box[0]]); _plt.plot(fit_box_closed[:, 0], fit_box_closed[:, 1], '-g', lw=2)
            except Exception:
                pass
            _plt.gca().set_aspect('equal', 'box')
            out = f'diagnostics/{base}_final_overlay.png'
            _plt.title(f'{folder} final overlay (auto-tuned)')
            _plt.tight_layout(); _plt.savefig(out, dpi=150, bbox_inches='tight'); _plt.close()
            print('Saved overlay ->', out)

            # Attempt texture projection (best-effort)
            try:
                K, _, _ = parse_blender_intrinsics(py_file)
                colors = project_textures_to_mesh(mesh, mov_file, traj, K, video_clap_frame)
                colors_path = f'diagnostics/{base}_mesh_colors.npy'
                _np.save(colors_path, _np.array(colors))
                print('Saved mesh colors ->', colors_path)
            except Exception as e:
                print('Texture projection failed:', e)

            summary.append({'folder': folder, 'mesh': mesh_path, 'overlay': out, 'params': best_params, 'debug': dbg})

        except Exception as e:
            print('Auto-tune failed for', folder, e)

    return pd.DataFrame(summary)

# NOTE: Do not auto-run this cell automatically to avoid long computations without explicit consent.
# The following lines execute the auto-tune now per your confirmation.
print('Starting auto-tune and apply (this may take several minutes)...')
summary_df = auto_tune_and_apply()
summary_df.to_csv('diagnostics/auto_tune_summary.csv', index=False)
print('Auto-tune summary saved -> diagnostics/auto_tune_summary.csv')
print(summary_df)


Starting auto-tune and apply (this may take several minutes)...
Saved best params -> diagnostics/TestData1_best_params.json


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/85562880.py:41: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_file)


generate_balanced_room_mesh input point count: 18700
RANSAC-clean point count: 15582
DBSCAN clusters found: 5 (noise points: 1)
minAreaRect dimensions: width=4.792, depth=3.679
Saved mesh -> diagnostics/TestData1_final_mesh.npy
Saved overlay -> diagnostics/TestData1_final_overlay.png
Saved mesh colors -> diagnostics/TestData1_mesh_colors.npy
Saved best params -> diagnostics/TestData2_best_params.json


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_33832/85562880.py:41: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_file)


generate_balanced_room_mesh input point count: 40203
RANSAC-clean point count: 30841
DBSCAN clusters found: 5 (noise points: 3)
minAreaRect dimensions: width=5.714, depth=3.724
Saved mesh -> diagnostics/TestData2_final_mesh.npy
Saved overlay -> diagnostics/TestData2_final_overlay.png
Saved mesh colors -> diagnostics/TestData2_mesh_colors.npy
Auto-tune summary saved -> diagnostics/auto_tune_summary.csv
           folder                                  mesh  \
0  Data/TestData1  diagnostics/TestData1_final_mesh.npy   
1  Data/TestData2  diagnostics/TestData2_final_mesh.npy   

                                   overlay  \
0  diagnostics/TestData1_final_overlay.png   
1  diagnostics/TestData2_final_overlay.png   

                                              params  \
0  {'dbscan_eps': 0.18, 'cluster_keep_ratio': 0.3...   
1  {'dbscan_eps': 0.24, 'cluster_keep_ratio': 0.3...   

                                               debug  
0  {'input_point_count': 18700, 'ransac_point_cou...  